In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
from openai import OpenAI
import os
api_key = os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [22]:
models = openai_client.models.list()

for model in models.data:
    print(model.id)

meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
allam-2-7b
openai/gpt-oss-safeguard-20b
groq/compound-mini
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
groq/compound
qwen/qwen3.8-27b
openai/gpt-oss-20b
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi


In [31]:
def llm(prompt):
    response = openai_client.responses.create(
        model="openai/gpt-oss-120b",
        input=prompt)
    return response.output_text

In [34]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely! We’re glad you’ve found the course and would love to have you join.

**Here’s how you can get started right away:**

1. **Check the enrollment window** – Our courses typically stay open for enrollment until the start date, and we’re still accepting participants for the upcoming session.  
2. **Confirm you meet any prerequisites** – Most of our courses are open to all skill levels, but if this one has a specific requirement (e.g., a basic familiarity with X), we’ll let you know when you begin the sign‑up process.  
3. **Provide your contact details** – We’ll need a valid email address (and optionally a phone number) so we can send you the confirmation, login credentials, and any pre‑course materials.  
4. **Complete the short registration form** – Once we have your email, we’ll send you a link to a secure form where you can:
   * Confirm your personal information  
   * Choose your preferred payment method (if the course isn’t free)  
   * Agree to the terms of participation

In [35]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [36]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [37]:
answer = llm(prompt)
print(answer)

Yes—you can still join the course. Just keep in mind that if you want a certificate, you’ll need to submit your project before the submission deadline.


**What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.**

In code, it looks like this:

``
 def rag(question):

    search_results = search(question)

    user_prompt = build_prompt(question, search_results)

    return llm(user_prompt)
    ``

That's the entire architecture. It comes down to three components.

The pieces are 
* search, 
* the prompt, 
and 
* the LLM:

search
prompt
LLM

**The LLM only sees the documents we hand it, so its answers are grounded in our data. If the right document is retrieved, the answer is good. If it's not, the LLM gets the wrong context and the answer is wrong. Your model is only as good as your retrieval, so search quality matters a lot for RAG.**

In [43]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
response.raise_for_status()
courses_raw = response.json()


In [45]:
documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)
len(documents)
documents[0]


{'id': '4487db3924',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I access the course modules and materials?',
 'answer': "All course materials are in the GitHub repo. Each module has its own folder (e.g. `01-overview`, `03-mcp`), and cohort-specific homework is under the `cohorts/` folder.\n\nFor the 2026 cohort, the week-by-week folders are:\n\n- Week 1: [01-ai-native-workflow](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/01-ai-native-workflow)\n- Week 2: [02-development](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/02-development)\n- Week 3: [03-deployment](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/03-deployment)\n- Week 4: [04-devops](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/04-devops)\n\nStart from the folder for your current week — it gathers everything for that week in one place.\n\n